In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import re
from sklearn.cluster import KMeans
import matplotlib.ticker as ticker

sns.set_style("whitegrid")
sns.set_context("talk", font_scale=1.1)

# Set the plot style for consistency
SAVE_PATH = "plots/"
os.makedirs(SAVE_PATH, exist_ok=True)

## 1. Load data

In [2]:
replace_dict = {
    'strongsort': 'StrongSORT',
    'ocsort': 'OC-SORT',
    'bytetrack': 'ByteTrack',
    'botsort': 'BoT-SORT',
    'deepocsort': 'Deep OC-SORT',
    'imprassoc': 'ImprAssOC'
}

In [3]:
# Load FPS results for GPU (3080 Ti) and TPU from CSVs
df_gpu_320 = pd.read_csv("fps_3080ti_320_results.csv")
df_gpu_320.yolo_model = df_gpu_320.yolo_model.str.replace('.pt', '')
df_gpu_512 = pd.read_csv("fps_3080ti_512_results.csv")
df_gpu_512.yolo_model = df_gpu_512.yolo_model.str.replace('.pt', '')
df_tpu_320 = pd.read_csv("fps_tpu_320_results.csv")
df_tpu_320.yolo_model = df_tpu_320.yolo_model.str.replace("_full_integer_quant_edgetpu.tflite", '')
df_tpu_320.yolo_model = df_tpu_320.yolo_model.str.replace("../tpu_weights/v8/320/", '')
df_tpu_512 = pd.read_csv("fps_tpu_512_results.csv")
df_tpu_512.yolo_model = df_tpu_512.yolo_model.str.replace("_full_integer_quant_edgetpu.tflite", '')
df_tpu_512.yolo_model = df_tpu_512.yolo_model.str.replace("../tpu_weights/v8/512/", '')

df_gpu_320["hardware"] = "GPU"
df_gpu_320["img_size"] = 320
df_gpu_512["hardware"] = "GPU"
df_gpu_512["img_size"] = 512
df_tpu_320["hardware"] = "TPU"
df_tpu_320["img_size"] = 320
df_tpu_512["hardware"] = "TPU"
df_tpu_512["img_size"] = 512

df_fps = pd.concat([df_gpu_320, df_gpu_512, df_tpu_320, df_tpu_512], ignore_index=True)
df_fps.drop_duplicates(inplace=True)
df_fps = df_fps.drop(columns=['avg_time_per_frame'])
df_fps["reid_model"] = df_fps["reid_model"].str.split("_").str[:-1].str.join('_')
df_fps['tracker'] = df_fps['tracker'].replace(replace_dict)
print(len(df_fps))
print(df_fps.columns)
df_fps.sample(5)

3660
Index(['tracker', 'yolo_model', 'reid_model', 'object_count', 'avg_fps',
       'min_time_per_frame', 'max_time_per_frame', 'hardware', 'img_size'],
      dtype='object')


,tracker,yolo_model,reid_model,object_count,avg_fps,min_time_per_frame,max_time_per_frame,hardware,img_size
3205,StrongSORT,yolov8n,clip,1,1.662113,538.6,5562.6,TPU,512
423,OC-SORT,yolov8x,osnet_x0_25,4,84.759349,11.5,14.1,GPU,320
2383,Deep OC-SORT,yolov8m,osnet_ain_x1_0,4,51.457440,17.0,36.9,GPU,512
2100,BoT-SORT,yolov8s,clip,1,70.354268,11.6,1659.6,GPU,512
2044,BoT-SORT,yolov8n,osnet_x0_25,5,75.686483,12.0,20.0,GPU,512


In [4]:
benchmarks = []
for name in os.listdir('.'):
    if "results_" not in name:
        continue
    fps = int(name.split('_')[1][:-3])
    df = pd.read_csv(f"{name}/results.csv")
    df['fps_bench'] = fps
    benchmarks.append(df.copy())
df_benchmarks = pd.concat(benchmarks, ignore_index=True)
df_benchmarks = df_benchmarks.drop(columns=['FPS', 'Elapsed_time', 'Status'])
df_benchmarks.rename(columns={'Tracker': 'tracker', 'YOLO Model': 'yolo_model', "REID Model": "reid_model", "ImgSz": 'img_size', "HOTA": 'hota',  "MOTA": 'mota',  "IDF1": 'idf1'}, inplace=True)

df_benchmarks["yolo_model"] = df_benchmarks["yolo_model"].str.split("_").str[0]
df_benchmarks["reid_model"] = df_benchmarks["reid_model"].str.split("_").str[:-1].str.join('_')
df_benchmarks['tracker'] = df_benchmarks['tracker'].replace(replace_dict)

df_benchmarks.drop_duplicates(inplace=True)
print(len(df_benchmarks))
print(df_benchmarks.columns)
df_benchmarks.sample(5)

2880
Index(['tracker', 'reid_model', 'yolo_model', 'img_size', 'hota', 'mota',
       'idf1', 'fps_bench'],
      dtype='object')


,tracker,reid_model,yolo_model,img_size,hota,mota,idf1,fps_bench
1852,Deep OC-SORT,lmbn_n,yolov8m,512,44.088,48.026,58.125,30
2750,BoT-SORT,lmbn_n,yolov8n,512,38.704,43.303,51.334,5
1489,StrongSORT,clip,yolov8x,320,44.613,44.572,61.416,30
473,ImprAssOC,osnet_ain_x1_0,yolov8l,512,40.014,49.910,50.089,15
974,StrongSORT,osnet_x0_75,yolov8x,512,42.121,43.527,57.006,3


In [5]:
import pandas as pd

# Create a copy of df_fps and rename 'avg_fps' to 'fps_eval'
df_fps_copy = df_fps.copy().rename(columns={'avg_fps': 'fps_eval'})

# Add an index column to track each row
df_fps_copy = df_fps_copy.reset_index().rename(columns={'index': 'fps_index'})

# Merge with df_benchmarks on the common keys; include 'img_size' if needed
merged = pd.merge(
    df_fps_copy,
    df_benchmarks,
    on=['tracker', 'yolo_model', 'reid_model', 'img_size'],
    suffixes=('', '_bench')  # Only benchmark has 'fps'
)

# Compute the absolute difference between df_fps's fps_eval and benchmark's fps
merged['diff'] = (merged['fps_eval'] - merged['fps_bench']).abs()

# For each original df_fps row (identified by fps_index), choose the benchmark row with the smallest fps difference
best_matches = merged.sort_values('diff').groupby('fps_index', as_index=False).first()

# Rename the benchmark's fps column to 'fps_benсh'
# best_matches = best_matches.rename(columns={'fps': 'fps_bench'})

# Merge back the selected benchmark columns ('hota', 'mota', 'idf1', and 'fps_benсh') to the original df_fps_copy
df_all = pd.merge(
    df_fps_copy,
    best_matches[['fps_index', 'fps_bench', 'hota', 'mota', 'idf1']],
    on='fps_index',
    how='left'
)

# Optionally, drop the temporary 'fps_index' column
df_all = df_all.drop(columns='fps_index')

# Now df_all is a copy of df_fps (with 'fps_eval' instead of 'avg_fps') 
# and with added benchmark columns, where the benchmark's fps column is renamed to 'fps_benсh'
df_all[df_all.fps_eval <30].sample(5)

,tracker,yolo_model,reid_model,object_count,fps_eval,min_time_per_frame,max_time_per_frame,hardware,img_size,fps_bench,hota,mota,idf1
2997,BoT-SORT,yolov8s,osnet_x0_25,3,4.626630,205.1,256.3,TPU,320,5,38.498,41.527,52.320
2820,OC-SORT,yolov8s,osnet_x1_0,1,24.029071,35.1,72.5,TPU,320,30,38.265,40.245,51.373
2915,ByteTrack,yolov8s,osnet_x0_25,1,29.221736,33.4,70.2,TPU,320,30,38.979,41.386,52.651
3174,ImprAssOC,yolov8s,osnet_ibn_x1_0,5,0.944975,1036.6,1219.2,TPU,320,1,33.875,32.199,42.488
2234,BoT-SORT,yolov8x,lmbn_n,5,26.400510,32.2,50.6,GPU,512,30,46.752,50.154,62.040


In [6]:
print('tracker', df_all.tracker.unique())
print('yolo_model', df_all.yolo_model.unique())
print('reid_model', df_all.reid_model.unique())
print('img_size', df_all.img_size.unique())


tracker ['StrongSORT' 'OC-SORT' 'ByteTrack' 'BoT-SORT' 'Deep OC-SORT' 'ImprAssOC']
yolo_model ['yolov8n' 'yolov8s' 'yolov8m' 'yolov8l' 'yolov8x']
reid_model ['osnet_x1_0' 'osnet_x0_75' 'osnet_x0_5' 'osnet_x0_25' 'clip' 'lmbn_n'
 'osnet_ibn_x1_0' 'osnet_ain_x1_0']
img_size [320 512]


## 2. Benchmarks analyze 

In [7]:
metrics = ['hota', 'mota', 'idf1']

### Metric = Metric(yolo_size)

In [8]:
yolo_models = df_all.yolo_model.unique()  # Order as in the dataframe or set a desired order
ordered_yolo = ['yolov8n', 'yolov8s', 'yolov8m', 'yolov8l', 'yolov8x']
img_sizes = df_all.img_size.unique()
trackers = df_all.tracker.unique()
metrics = ['hota', 'mota', 'idf1']

cur_save_path = SAVE_PATH + "yolo_size_vs_metric/"
os.makedirs(cur_save_path, exist_ok=True)

for tracker in trackers:
    # Create a grid: rows = metrics (3), columns = image sizes
    fig, axes = plt.subplots(nrows=len(metrics), ncols=len(img_sizes), figsize=(18, 12), 
                             sharex=True, sharey='row')
    
    for row, metric in enumerate(metrics):
        for col, img_size in enumerate(img_sizes):
            ax = axes[row, col]
            # Filter and group by YOLO model for the given tracker and image size.
            grouped = df_benchmarks[
                (df_benchmarks.img_size == img_size) &
                (df_benchmarks.tracker == tracker)
            ].groupby(['yolo_model'])[metrics].mean().reset_index()
            
            grouped['yolo_model'] = pd.Categorical(grouped['yolo_model'], categories=ordered_yolo, ordered=True)
            grouped = grouped.sort_values('yolo_model')
            
            # Plot using YOLO model as categorical x-axis.
            sns.lineplot(
                data=grouped,
                x='yolo_model',
                y=metric,
                marker='o',
                sort=False,  # maintain order in grouped data
                ax=ax
            )
            
            # Titles and axis labels
            if row == 0:
                ax.set_title(f'Размер: {img_size}x{img_size}', fontsize=30)
            if row == len(metrics) - 1:
                ax.set_xlabel('Версия YOLO', fontsize=35)
            else:
                ax.set_xlabel('')
            ax.set_ylabel(metric.upper(), fontsize=30)
            ax.tick_params(axis='both', which='major', labelsize=25)
            
            # Set x-ticks: positions 0,1,2,... corresponding to YOLO model names.
            unique_ticks = grouped['yolo_model'].tolist()
            ax.set_xticks(range(len(unique_ticks)))
            ax.set_xticklabels(unique_ticks, rotation=30, fontsize=25)
            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
            ax.grid(True, which='both', linestyle='--', linewidth=0.75)
    
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    fig.savefig(cur_save_path + f'{tracker}.png')
    plt.close(fig)


In [9]:
# Define the desired orders.
ordered_yolo_subset = ['yolov8n', 'yolov8x']
ordered_img_sizes = sorted(df_all.img_size.unique())  # e.g., [320, 512]

cur_save_path = os.path.join(SAVE_PATH, "yolo_size_vs_metric")
os.makedirs(cur_save_path, exist_ok=True)

for metric in metrics:
    # Filter the data for only the two YOLO models and select only necessary columns.
    df_subset = df_benchmarks[df_benchmarks.yolo_model.isin(ordered_yolo_subset)][['tracker', 'img_size', 'yolo_model', metric]].copy()
    
    # Group by tracker, image size, and YOLO model; take the maximum value for the metric.
    grouped = df_subset.groupby(['tracker', 'img_size', 'yolo_model'])[metric].mean().reset_index()
    
    # Force the ordering of yolo_model.
    grouped['yolo_model'] = pd.Categorical(grouped['yolo_model'], categories=ordered_yolo_subset, ordered=True)
    grouped = grouped.sort_values(['tracker', 'img_size', 'yolo_model'])
    
    # Pivot the table so that rows are trackers and columns are a MultiIndex: (img_size, yolo_model).
    pivot_table = grouped.pivot(index='tracker', columns=['img_size', 'yolo_model'], values=metric)
    
    # Ensure columns are ordered as desired.
    col_index = pd.MultiIndex.from_product([ordered_img_sizes, ordered_yolo_subset], names=['img_size', 'yolo_model'])
    pivot_table = pivot_table.reindex(columns=col_index)
    
    # Round numeric values to one decimal and replace missing values with a dash.
    pivot_table = pivot_table.round(1).fillna("-")
    
    # Convert the pivot table to LaTeX code.
    latex_code = pivot_table.to_latex(multicolumn=True, multirow=True, escape=True, na_rep="-", float_format="%.1f")
    
    # Replace header names if needed.
    latex_code = latex_code.replace("tracker", "Алгоритм")
    latex_code = latex_code.replace("yolo_model", "Детектор")
    latex_code = latex_code.replace("img_size", "Размер изображения")
    
    # --- Post-process the header to merge pairs of columns for each image size ---
    # First, merge two adjacent \multicolumn{1}{c}{<img_size>} entries into one \multicolumn{2}{c}{<img_size>}
    lines = latex_code.splitlines()
    new_lines = []
    for line in lines:
        new_line = re.sub(r'(\\multicolumn\{1\}\{c\}\{(\d+)\})\s*&\s*(\\multicolumn\{1\}\{c\}\{\2\})', r'\\multicolumn{2}{c}{\2}', line)
        new_lines.append(new_line)
    # Now, find the index of the first occurrence of "\midrule"
    try:
        i_mid = next(i for i, line in enumerate(new_lines) if line.strip() == "\\midrule")
    except StopIteration:
        i_mid = 3  # fallback, if not found
    
    # Build custom header rows.
    header1 = " & " + " & ".join([f"\\multicolumn{{2}}{{c}}{{{img_size}}}" for img_size in ordered_img_sizes]) + " \\\\"
    header2 = " "  # empty cell for the leftmost column (tracker names)
    for img_size in ordered_img_sizes:
        for yolo in ordered_yolo_subset:
            header2 += f" & {yolo}"
    header2 += " \\\\"
    # Replace all lines between \toprule and \midrule with our header rows.
    new_lines = new_lines[:1] + [header1, header2, "\\midrule"] + new_lines[i_mid+1:]
    
    latex_code_modified = "\n".join(new_lines)
    # Replace any remaining {r} with {c} if necessary.
    latex_code_modified = latex_code_modified.replace("{r}", "{c}")
    # -----------------------------------------
    
    caption = f"Среднее значение метрики {metric.upper()} для yolov8n и yolov8x"
    label = f"tab:mean_{metric}_yolo_size"
    
    # Wrap the LaTeX table in a table environment with caption and label.
    full_latex = (
        "\\begin{table}[htbp]\n"
        f"\n\\caption{{{caption}}}\n" +
        f"\\label{{{label}}}\n" +
        "\\centering\n" +
        latex_code_modified +
        "\\end{table}"
    )
    
    # Save the LaTeX code to a file.
    filename = f"mean_{metric}_yolo_size.tex"
    full_latex = full_latex.replace("{r}", "{c}")
    with open(os.path.join(cur_save_path, filename), "w") as f:
        f.write(full_latex)
    
    print(f"Saved LaTeX table for {metric} as {filename}")

Saved LaTeX table for hota as mean_hota_yolo_size.tex
Saved LaTeX table for mota as mean_mota_yolo_size.tex
Saved LaTeX table for idf1 as mean_idf1_yolo_size.tex


### Metric = Metric(yolo size, reid_model)

In [10]:

yolo_models = df_all.yolo_model.unique()
# yolo_models = ["yolov8n", "yolov8s", "yolov8x"]
img_sizes = df_all.img_size.unique()
trackers = df_all.tracker.unique()
reid_models = df_all.reid_model.unique()
reid_models = [x for x in reid_models if 'ibn' not in x and 'ain' not in x]
reid_models = sorted(reid_models, key=lambda x: len(x), reverse=True)

cur_save_path = SAVE_PATH + "yolo_size_and_reid_vs_metric/"
os.makedirs(cur_save_path, exist_ok=True)

for tracker in trackers:
    # Create a grid: rows = metrics, columns = image sizes
    fig, axes = plt.subplots(nrows=len(metrics), ncols=len(img_sizes), figsize=(18, 12),
                             sharex=True, sharey='row')
    
    for row, metric in enumerate(metrics):
        for col, img_size in enumerate(img_sizes):
            ax = axes[row, col]
            # Loop over each YOLO model to plot its line
            for yolo_model in yolo_models:
                # Group by 'reid_model' (the x-axis now) and compute mean for the given metrics.
                grouped = df_benchmarks[
                    (df_benchmarks.img_size == img_size) &
                    (df_benchmarks.yolo_model == yolo_model) &
                    (df_benchmarks.tracker == tracker) & 
                    (df_benchmarks.reid_model.isin(reid_models))
                ].groupby(['reid_model'])[metrics].mean().reset_index()
                
                sns.lineplot(
                    data=grouped,
                    x='reid_model',
                    y=metric,
                    marker='o',
                    label=yolo_model,
                    ax=ax
                )
            # Titles and labels
            if row == 0:
                ax.set_title(f'Размер: {img_size}x{img_size}', fontsize=30)
            if row == len(metrics) - 1:
                ax.set_xlabel('ReID Model', fontsize=35)
            else:
                ax.set_xlabel('')
            ax.set_ylabel(metric.upper(), fontsize=30)
            ax.tick_params(axis='both', which='major', labelsize=25)

            # For categorical x-axis, we assign positions 0,1,2,...
            ax.set_xticks(range(len(reid_models)))
            ax.set_xticklabels(reid_models, rotation=30, fontsize=25)
            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
            ax.grid(True, which='both', linestyle='--', linewidth=0.75)
    
    # Remove individual legends from subplots
    handles, labels = axes[0, 0].get_legend_handles_labels()
    for ax in axes.flatten():
        if ax.get_legend() is not None:
            ax.get_legend().remove()
    
    # Add one common legend at the bottom center.
    fig.legend(handles, labels, loc='lower center', ncol=len(yolo_models),
               title='Детектор', fontsize=30, title_fontsize=30)
    
    fig.tight_layout(rect=[0, 0.12, 1, 1])
    fig.savefig(cur_save_path + f'{tracker}.png')
    plt.close(fig)

### Metric of method = Metric(FPS Benchmark, yolo size)

In [11]:
fps_data = df_fps[df_fps.avg_fps < 16].avg_fps.values.reshape(-1, 1)
n_clusters = 5 
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans.fit_predict(fps_data)
print(sorted(kmeans.cluster_centers_.reshape(-1).astype(int)))

[1, 3, 5, 11, 14]


In [12]:
yolo_models = ["yolov8n", "yolov8s", "yolov8x"]
img_sizes = df_all.img_size.unique()
trackers = df_all.tracker.unique()

cur_save_path = SAVE_PATH + "fps_vs_metric/"
os.makedirs(cur_save_path, exist_ok=True)

for tracker in trackers:
    # Create a 3 (rows: metrics) x 2 (cols: image sizes) grid
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(20, 10), sharex=True, sharey='row')
    
    for row, metric in enumerate(metrics):
        for col, img_size in enumerate(img_sizes):
            ax = axes[row, col]
            for yolo_model in yolo_models:
                grouped = df_benchmarks[
                    (df_benchmarks.img_size == img_size) &
                    (df_benchmarks.yolo_model == yolo_model) &
                    (df_benchmarks.tracker == tracker)
                ].groupby(['fps_bench'])[metrics].mean().reset_index()
                
                sns.lineplot(
                    data=grouped,
                    x='fps_bench',
                    y=metric,
                    marker='o',
                    label=yolo_model,
                    ax=ax
                )
            if row == 0:
                ax.set_title(f'Размер: {img_size}x{img_size}', fontsize=30)
            if row == 2:
                ax.set_xlabel('Частота кадров', fontsize=30)
            else:
                ax.set_xlabel('')
            ax.set_ylabel(metric.upper(), fontsize=30)
            ax.tick_params(axis='both', which='major', labelsize=25)
            
            # Set x-ticks based on the unique fps_bench values for the current tracker and img_size.
            unique_ticks = sorted(
                df_benchmarks[
                    (df_benchmarks.img_size == img_size) &
                    (df_benchmarks.tracker == tracker)
                ]['fps_bench'].unique()
            )
            ax.set_xticks(unique_ticks)
            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
            
            # Enable a dense grid
            ax.grid(True, which='both', linestyle='--', linewidth=0.75)
    
    # Remove individual legends from each subplot.
    handles, labels = axes[0, 0].get_legend_handles_labels()
    for ax in axes.flatten():
        if ax.get_legend() is not None:
            ax.get_legend().remove()
    
    # Add a single common legend at the bottom center.
    fig.legend(handles, labels, loc='lower center', ncol=len(yolo_models), title='Детектор',
               fontsize=30, title_fontsize=30)
    
    # fig.suptitle(f'Влияние частоты кадров на метрики для алгоритма {tracker}', fontsize=24, y=0.98)
    fig.tight_layout(rect=[0, 0.12, 1, 1])
    fig.savefig(cur_save_path + f'{tracker}.png')
    plt.close(fig)

In [13]:
sns.boxplot(x='tracker', y='mota', data=df_all[df_all['fps_bench'] == 30])
plt.title('Distribution of MOTA by Tracker (30 fps Bench)')
plt.xlabel('Tracker')
plt.ylabel('MOTA')
plt.xticks(rotation=45)
# plt.show()
plt.close()